> **Exploratory training notebook only.** Do not use this as the release model pipeline. Run `train_model.py` and `audit_model.py` instead.


# 04 — Machine Learning Model Experiments
**Goal:** Compare dummy majority/prior baseline, logistic regression and random forest to predict **regulation-time** outcomes: `team_1_win`, `draw`, `team_2_win`.

**Time-based evaluation:** train on 2002–2018, validate on 2022, then refit the selected algorithm through 2022 and **test only once** on 2026. This is a small, World Cup-only dataset; report limitations and never claim these probabilities are guaranteed accurate.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'raw').exists():
    raise FileNotFoundError('Open this notebook with working directory at the project root or notebooks folder.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_REPORTS = PROJECT_ROOT / 'data' / 'reports'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_REPORTS.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report, confusion_matrix
from src.worldcup_data import FEATURE_COLUMNS

model_df=pd.read_csv(DATA_PROCESSED/'prematch_features.csv',parse_dates=['date'])
model_df['year']=model_df['year'].astype(int)
train=model_df[(model_df['year']>=2002)&(model_df['year']<=2018)].copy()
val=model_df[model_df['year']==2022].copy()
test=model_df[model_df['year']==2026].copy()
assert len(train)>0 and len(val)>0 and len(test)>0, 'Expected 2002–2018, 2022, and 2026 records'
assert train['date'].max()<val['date'].min()<test['date'].min()
labels=['team_1_win','draw','team_2_win']
print('Train:',train['year'].unique(),'n=',len(train))
print('Validation:',val['year'].unique(),'n=',len(val))
print('Test:',test['year'].unique(),'n=',len(test))
print('Training class balance:')
display(train['result_90'].value_counts())

## A. Candidate models
Impute missing values **inside** the training pipeline so that validation/test information is never used to calculate replacements. Use one fixed feature set and a simple baseline.

In [ ]:
models={
    'Dummy Prior':Pipeline([
        ('impute',SimpleImputer(strategy='constant',fill_value=0)),
        ('model',DummyClassifier(strategy='prior'))
    ]),
    'Logistic Regression':Pipeline([
        ('impute',SimpleImputer(strategy='median')),
        ('scale',StandardScaler()),
        ('model',LogisticRegression(max_iter=1500,random_state=42))
    ]),
    'Random Forest':Pipeline([
        ('impute',SimpleImputer(strategy='median')),
        ('model',RandomForestClassifier(n_estimators=200,max_depth=5,
                                        min_samples_leaf=5,random_state=42))
    ])
}

def report_metrics(estimator,df):
    X=df[FEATURE_COLUMNS]; y=df['result_90']
    pred=estimator.predict(X)
    prob=estimator.predict_proba(X)
    return {'accuracy':accuracy_score(y,pred),
            'macro_f1':f1_score(y,pred,labels=labels,average='macro',zero_division=0),
            'log_loss':log_loss(y,prob,labels=estimator.classes_)}

## B. Validation experiments (2022)
Select the model using 2022 validation performance. Do not choose a model based on the 2026 test scores.

In [ ]:
validation_rows=[]
for name,estimator in models.items():
    estimator.fit(train[FEATURE_COLUMNS],train['result_90'])
    metrics=report_metrics(estimator,val)
    validation_rows.append({'model':name,**metrics})
validation_df=(pd.DataFrame(validation_rows)
    .sort_values(['macro_f1','log_loss'],ascending=[False,True]).reset_index(drop=True))
display(validation_df)
selected_name=validation_df.iloc[0]['model']
print('Selected based on VALIDATION macro-F1:',selected_name)

## C. Refit on pre-2026 data, then evaluate 2026 once
Prediction uses prior World Cup results only. Within the 2026 competition, earlier **completed matches** may be part of the form for later matches. This is a walk-forward-style historical backtest, not a forecast made before the entire 2026 tournament.

In [ ]:
training_full=pd.concat([train,val],ignore_index=True)
selected=models[selected_name]
selected.fit(training_full[FEATURE_COLUMNS],training_full['result_90'])
test_metrics=report_metrics(selected,test)
print('Holdout TEST (2026):',test_metrics)
pred=selected.predict(test[FEATURE_COLUMNS])
print(classification_report(test['result_90'],pred,labels=labels,zero_division=0))
print('Confusion matrix: rows=actual, cols=predicted')
print(pd.DataFrame(confusion_matrix(test['result_90'],pred,labels=labels),
                   index=labels,columns=labels))

## D. Export model and held-out predictions
The dashboard can read these predictions. They are retrospective model estimates of 2026 matches, **not future matches** or odds recommendations.

In [ ]:
# This notebook is EXPLORATORY ONLY and must never overwrite the audited model/reports.
# Run `python train_model.py` then `python audit_model.py` for production artifacts.
print('Exploratory test metrics (not production):', test_metrics)
print('For reproducible release reports, run train_model.py and audit_model.py.')
